In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive/MS/rag_code_audit/

/content/drive/MyDrive/MS/rag_code_audit


In [ ]:
pip install -q langgraph openai chromadb pydantic sentence-transformers pandas scikit-learn pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

# Storing pdf into Vector DB



In [ ]:
import pypdf
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="nist_ai_rmf_full")


pdf_path = "nist_ai_rmf.pdf"
reader = pypdf.PdfReader(pdf_path)

full_text = ""
print(f"Reading {len(reader.pages)} pages from NIST AI RMF PDF...")
for page in reader.pages:
    text = page.extract_text()
    if text:
        full_text += text + " "

raw_paragraphs = full_text.split(".\n")

Reading 48 pages from NIST AI RMF PDF...


In [ ]:
documents = []
ids = []

for i, raw_p in enumerate(raw_paragraphs):
    clean_paragraph = " ".join(raw_p.split())

    if len(clean_paragraph) > 50:
        documents.append(clean_paragraph)
        ids.append(f"paragraph_{len(documents)}")

collection.add(
    documents=documents,
    ids=ids
)
print(f"Successfully stored {len(documents)} semantic paragraph chunks from the NIST PDF into ChromaDB!")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 56.2MiB/s]


Successfully stored 288 semantic paragraph chunks from the NIST PDF into ChromaDB!


In [ ]:
results = collection.query(query_texts=["What are the challenges of AI risk management"], n_results=5)
results

{'ids': [['paragraph_25',
   'paragraph_39',
   'paragraph_21',
   'paragraph_17',
   'paragraph_23']],
 'embeddings': None,
 'documents': [['1.2 Challenges for AI Risk Management Several challenges are described below. They should be taken into account when managing risks in pursuit of AI trustworthiness',
   'To the extent that challenges for specifying AI risk tolerances remain unresolved, there may be contexts where a risk management framework is not yet readily applicable for mitigating negative AI risks',
   'While risk management processes generally address negative impacts, this Framework of- fers approaches to minimize anticipated negative impacts of AI systems and identify op- portunities to maximize positive impacts. Effectively managing the risk of potential harms could lead to more trustworthy AI systems and unleash potential benefits to people (individ- uals, communities, and society), organizations, and systems/ecosystems. Risk management can enable AI developers and use

In [ ]:
results = collection.query(query_texts=["extract all the policies and conditions required to be checked for model development"], n_results=10)
results1 = collection.query(query_texts=["extract all the policies and conditions required to be checked for the model development code"], n_results=10)
flattened_rules = "\n".join(results['documents'][0])

In [ ]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="nist_ai_rmf_full")


pdf_path = "nist_ai_rmf.pdf"
reader = pypdf.PdfReader(pdf_path)

full_text = ""
print(f"Reading {len(reader.pages)} pages from NIST AI RMF PDF...")
for page in reader.pages:
    text = page.extract_text()
    if text:
        full_text += text + " "

raw_paragraphs = full_text.split(".\n")

documents = []
ids = []

for i, raw_p in enumerate(raw_paragraphs):
    clean_paragraph = " ".join(raw_p.split())

    if len(clean_paragraph) > 50:
        documents.append(clean_paragraph)
        ids.append(f"paragraph_{len(documents)}")

collection.add(
    documents=documents,
    ids=ids
)
print(f"Successfully stored {len(documents)} semantic paragraph chunks from the NIST PDF into ChromaDB!")

Reading 48 pages from NIST AI RMF PDF...
Successfully stored 288 semantic paragraph chunks from the NIST PDF into ChromaDB!


# Policies Retrieval

In [ ]:
import sys, chromadb
from typing import TypedDict, List, Dict, Any
from openai import OpenAI
from langgraph.graph import StateGraph, END

# Initialize the OpenAI client pointing to your Mac's ngrok URL
client = OpenAI(
    base_url="https://backup-kerchief-aloof.ngrok-free.dev/v1", # The /v1 is required for Ollama's OpenAI compatibility
    api_key="ollama" # Required by the SDK, but ignored by Ollama
)

# Set your target local model (must match the model you pulled in Ollama)
TARGET_MODEL = "qwen2.5:7b-instruct"

# Legal Extractor Node
def policy(query_txt) -> Dict[str, Any]:
    results = collection.query(query_texts=[query_txt], n_results=5)
    flattened_rules = "\n".join(results['documents'][0])

    prompt = f"Policies:\n{flattened_rules}\nExtract policy rules and math condition if any."
    response = client.chat.completions.create(
        model=TARGET_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content


In [ ]:
response = policy("What are the challenges of AI risk management")

'### Policy Rules Extracted\n\n1. **Challenges for Specifying AI Risk Tolerances**:\n   - If challenges in specifying AI risk tolerances remain unresolved, a risk management framework may not be applicable or readily usable.\n   \n2. **Risk Management Framework**:\n   - The framework offers approaches to minimize anticipated negative impacts of AI systems and identify opportunities to maximize positive impacts.\n   - Effective risk management processes generally address negative impacts; the framework can help mitigate these impacts.\n\n3. **Beneficial Outcomes from Risk Management**:\n   - Effectively managing potential harms of AI systems could lead to more trustworthy AI systems, benefiting individuals, communities, and society.\n   - Risk management enables understanding of impact, accounting for model limitations, and improving overall system performance and trustworthiness.\n   - This can increase the likelihood that AI technologies will be used in beneficial ways.\n\n4. **Framin

In [ ]:
print(response)

### Policy Rules Extracted

1. **Challenges for Specifying AI Risk Tolerances**:
   - If challenges in specifying AI risk tolerances remain unresolved, a risk management framework may not be applicable or readily usable.
   
2. **Risk Management Framework**:
   - The framework offers approaches to minimize anticipated negative impacts of AI systems and identify opportunities to maximize positive impacts.
   - Effective risk management processes generally address negative impacts; the framework can help mitigate these impacts.

3. **Beneficial Outcomes from Risk Management**:
   - Effectively managing potential harms of AI systems could lead to more trustworthy AI systems, benefiting individuals, communities, and society.
   - Risk management enables understanding of impact, accounting for model limitations, and improving overall system performance and trustworthiness.
   - This can increase the likelihood that AI technologies will be used in beneficial ways.

4. **Framing Risk**:
   - 